# 🧬 Workshop: Protein Disorder Prediction with Machine Learning

**The Goal:** Build a machine learning model that can predict which parts of it are disordered and which parts are ordered.

**The Data:**
* **Sequences:** We have protein sequences in FASTA format.
* **Labels (Ground Truth):** We have a database (DisProt) that tells us which regions are actually disordered.

**The Plan:**
1.  **Load Data:** Read sequences and labels.
2.  **Feature Engineering:** Convert letters (amino acids) into numbers (vectors) that an ML model can understand.
3.  **Training:** Teach a model to distinguish ordered vs. disordered residues.
4.  **Evaluation:** Check if our model is actually learning or just guessing.


**Task:** We need a function to open these files, throw away the header, and stitch the sequence lines together into a single string.

### Step 1: Imports and variables we are going to use later. We are also downloading all disprot annotations





In [ ]:
import numpy as np
import copy

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from sklearn.metrics import balanced_accuracy_score
from sklearn.tree import DecisionTreeClassifier, plot_tree

%matplotlib inline
import matplotlib.pyplot as plt

test_accession = 'Q05344'

AMINO_ACIDS = sorted("ACDEFGHIKLMNPQRSTVWY")

In [ ]:
# All-in-one cell:
# 1. Download the current DisProt regions TSV
# 2. Extract disorder annotations
# 3. Download missing UniProt FASTA files
# 4. Build the annotations dictionary

import os
import requests
import pandas as pd
from collections import defaultdict

# Configuration
data_location = "/content/disprot_data"
seq_dir = f"{data_location}/sequences"
disprot_file = f"{data_location}/disprot.tsv"

os.makedirs(seq_dir, exist_ok=True)

# ------------------------------------------------------------------
# Step 1: Download DisProt regions TSV
# ------------------------------------------------------------------
print("Downloading DisProt regions TSV...")

url = "https://disprot.org/api/search?format=tsv&release=current&namespace=structural_state&type=region"

r = requests.get(url, timeout=120)
r.raise_for_status()

with open(disprot_file, "w") as f:
    f.write(r.text)

print("Saved:", disprot_file)

# ------------------------------------------------------------------
# Step 2: Load and filter disorder annotations
# ------------------------------------------------------------------
df = pd.read_csv(disprot_file, sep="\t")

print(f"Rows in DisProt file: {len(df):,}")
print("Columns:")
for i, col in enumerate(df.columns):
    print(i, col)

# Keep rows where the annotation term contains "disorder"
disorder_df = df[df["term_name"].str.contains("disorder", case=False, na=False)].copy()

print(f"\nDisorder rows: {len(disorder_df):,}")
print(f"Unique accessions with disorder: {disorder_df['acc'].nunique():,}")

# ------------------------------------------------------------------
# Step 3: Build annotations dictionary
# ------------------------------------------------------------------
annotations = defaultdict(list)

for _, row in disorder_df.iterrows():
    acc = row["acc"]
    start = int(row["start"])
    end = int(row["end"])
    annotations[acc].append((start, end))

print(f"\nProteins with disorder annotations: {len(annotations):,}")

# ------------------------------------------------------------------
# Step 4: Download missing UniProt FASTA files
# ------------------------------------------------------------------
to_download = [
    acc for acc in annotations
    if not os.path.isfile(f"{seq_dir}/{acc}.fasta")
]

print(f"FASTA files already present: {len(annotations) - len(to_download):,}")
print(f"FASTA files to download:     {len(to_download):,}")

session = requests.Session()

def download_uniprot_fasta(acc):
    out_fasta = f"{seq_dir}/{acc}.fasta"

    if os.path.isfile(out_fasta):
        return True

    url = f"https://rest.uniprot.org/uniprotkb/{acc}.fasta"

    try:
        r = session.get(url, timeout=30)
        r.raise_for_status()

        with open(out_fasta, "w") as f:
            f.write(r.text)

        return True

    except Exception as e:
        print(f"[WARNING] Failed to download {acc}: {e}")
        return False

failed = []

for i, acc in enumerate(to_download, start=1):
    if not download_uniprot_fasta(acc):
        failed.append(acc)

    if i % 50 == 0 or i == len(to_download):
        print(
            f"Downloaded {i:,}/{len(to_download):,} "
            f"({100*i/len(to_download):.1f}%)"
        )

# ------------------------------------------------------------------
# Step 5: Summary
# ------------------------------------------------------------------
print("\nFinished.")
print(f"Successful downloads: {len(to_download) - len(failed):,}")
print(f"Failed downloads:     {len(failed):,}")
print(f"Sequence directory:   {seq_dir}")
print(f"Proteins annotated:   {len(annotations):,}")

print("\nExample annotations:")
for acc in list(annotations.keys())[:5]:
    print(acc, annotations[acc])

# Example usage:
# test_accession = "Q05344"
# print(annotations[test_accession])

### Step 2: Loading the Answers (Labels)
To train a model, we need the "answers." We are using the **DisProt** database, which lists the specific start and end positions of disordered regions.

**The Code below:**
1.  Opens the database file (`disprot.tsv`).
2.  Filters for lines where the type is explicitly `disorder`.
3.  Stores the **start** and **end** coordinates for every protein ID.

In [ ]:
# Recreate disorder_df from the DisProt TSV if it is not already in memory

import os
import pandas as pd

# Location used earlier in the notebook
# data_location should already be defined, for example:
# data_location = "/content/disprot_data"

disprot_tsv = f"{data_location}/disprot.tsv"

if not os.path.isfile(disprot_tsv):
    raise FileNotFoundError(
        f"Cannot find {disprot_tsv}. Please run the DisProt download cell first."
    )

# Load the TSV if disorder_df is missing
if "disorder_df" not in globals():
    print("Loading DisProt data...")

    # Column names based on the file structure
    columns = [
        "acc",
        "name",
        "disorder_content",
        "organism",
        "ncbi_taxon_id",
        "disprot_id",
        "region_id",
        "start",
        "end",
        "term_namespace",
        "term",
        "term_name",
        "ec",
        "ec_name",
        "reference",
        "region_sequence",
        "confidence",
        "obsolete",
    ]

    df = pd.read_csv(
        disprot_tsv,
        sep="\t",
        header=None,
        names=columns,
        dtype=str,
    )

    # Keep only rows whose namespace is "Disorder"
    disorder_df = df[df["term_namespace"] == "Disorder"].copy()

    # Convert coordinates to integers
    disorder_df["start"] = disorder_df["start"].astype(int)
    disorder_df["end"] = disorder_df["end"].astype(int)

    print(f"Loaded {len(disorder_df):,} disorder regions")
    print(
        f"Unique proteins with disorder annotations: "
        f"{disorder_df['acc'].nunique():,}"
    )
else:
    print("disorder_df is already defined.")
    print(f"Rows: {len(disorder_df):,}")
    print(f"Unique proteins: {disorder_df['acc'].nunique():,}")

### Step 3: Feature Engineering (The Sliding Window)
Machine Learning models cannot read strings like `"MVLSE..."`. They need numbers.

**The Strategy: Sliding Window**
A residue's behavior is influenced by its neighbors. We will look at a small window (e.g., 15 residues) around each target amino acid.

1.  **Windowing:** We slide a window of size 15 across the sequence.
2.  **Vectorization:** Inside that window, we count the frequency of the 20 amino acids (A, C, D, E...).
    * *Example:* If a window has many hydrophobic residues (V, L, I), it might be ordered. If it has many charged residues (D, E, K), it might be disordered.
3.  **Result:**
    * `x`: Our features (frequency lists).
    * `y`: Our labels (0 for Ordered, 1 for Disordered).

In [ ]:
# Define this function before running the cell that iterates over annotations

!pip install biopython
from Bio import SeqIO
import os

def get_sequence_from_fasta(accession):
    """
    Load the amino acid sequence for a UniProt accession from:
        {data_location}/sequences/{ACCESSION}.fasta

    Returns:
        str: sequence as a string
        None: if the FASTA file does not exist or cannot be read
    """
    accession = accession.upper().strip()
    fasta_file = os.path.join(data_location, "sequences", f"{accession}.fasta")

    if not os.path.isfile(fasta_file):
        print(f"[WARNING] FASTA file not found for {accession}: {fasta_file}")
        return None

    try:
        record = next(SeqIO.parse(fasta_file, "fasta"))
        return str(record.seq)
    except Exception as e:
        print(f"[WARNING] Could not read {fasta_file}: {e}")
        return None

In [ ]:
def get_aa_frequencies(sequence):
    """
    Converts a protein sequence into a 20-length frequency vector.
    """

    seq_len = len(sequence)
    if seq_len == 0:
        return [0.0] * 20

    # Iterate through the fixed alphabet, not the sequence, to ensure
    # the vector is always length 20 with the correct order.
    vector = [sequence.count(aa) / seq_len for aa in AMINO_ACIDS]

    return vector

x, y = [], []
window = 15
num_proteins_to_use = 100
cntr = 0
for acc, regions in annotations.items():
    seq = get_sequence_from_fasta(acc)
    if not seq:
        continue
    binary_class = [0] * len(seq)
    for s, e in regions:
        for i in range(s-1, e):
            binary_class[i] = 1

    for i in range(len(seq)):
        stretch = seq[max(0, i-window) : min(len(seq), i+window)]
        x.append(get_aa_frequencies(stretch))
        y.append(binary_class[i])
    cntr += 1
    if cntr >= num_proteins_to_use:
        break

print('Reading done!')

### Step 4: Training a Linear Classifier
We will start with a simple model: **Logistic Regression**. This model draws a line (or hyperplane) to separate the two classes.

**Key Concepts in the Code:**
* **Train/Test Split:** We hide 20% of the data (`test_size=0.2`) to test the model later. We never test on data the model has already seen!
* **Stratify:** Biological data is often **imbalanced** (there are way more "Ordered" residues than "Disordered" ones). `stratify=y` ensures our Training and Test sets have the same percentage of disorder, so the test is fair (or is it?).

In [ ]:
# Convert to numpy arrays for efficiency with sklearn
X_data = np.array(x)
y_data = np.array(y)


# --- Train/Test Split ---
# split into 80% train, 20% test
# stratify=y ensures the ratio of 0s and 1s is kept consistent in both sets
X_train, X_test, y_train, y_test = train_test_split(
    X_data, y_data, test_size=0.2, random_state=42, stratify=y_data
)

print(f"Total residues processed: {len(y_data)}")
print(f"Class balance: {np.sum(y_data == 0)} ordered vs {np.sum(y_data == 1)} disordered")

# --- Train Logistic Regression ---
print("Training Logistic Regression...")
# max_iter is increased because default (100) is often too low for convergence
clf = LogisticRegression(random_state=42, max_iter=1000)
clf.fit(X_data, y_data)

# --- Evaluation ---
print("Evaluating...")
y_pred = clf.predict(X_data)

# Basic Accuracy
acc = accuracy_score(y_data, y_pred)
print(f"Accuracy: {acc:.4f}")

### Step 5: The Accuracy Trap
Now we evaluate the model. Be careful!

* **Accuracy** is just `(Correct Predictions) / (Total Data)`.
* **The Trap:** If 90% of your data is "Ordered", a model that acts like a broken record and *always* guesses "Ordered" will still have 90% accuracy!

**Look closely at:**
* **Confusion Matrix:** shows exactly where we made mistakes.
* **Recall (Sensitivity):** How many of the actual disordered residues did we find?
* **Balanced Accuracy:** The average recall of both classes. This is the real metric to trust.

In [ ]:
# Balanced Accuracy
acc = balanced_accuracy_score(y_data, y_pred)
print(f"Balanced accuracy: {acc:.4f}")
# Detailed Report (Precision, Recall, F1-Score)
print("\nClassification Report:")
print(classification_report(y_data, y_pred, target_names=['Ordered (0)', 'Disordered (1)']))

# Confusion Matrix
tn, fp, fn, tp = confusion_matrix(y_data, y_pred).ravel().tolist()
print("Confusion Matrix:")
print(f"{'':>15} Predicted Neg | Predicted Pos")
print(f"{'Actual Neg':>15} [ {tn:^11} | {fp:^11} ]")
print(f"{'-'*45}")
print(f"{'Actual Pos':>15} [ {fn:^11} | {tp:^11} ]")

How do you think our model is performing? Do we have any pitfalls?

### Step 6: Trying a Non-Linear Model
The linear model struggled (likely predicting "Ordered" too often). Let's try a **Decision Tree**.

* **Why?** Decision trees can capture complex, non-linear rules (e.g., *"If Proline is high AND Cysteine is low..."*).
* **Max Depth:** We limit the depth to prevent **overfitting** (memorizing the training data instead of learning general rules).

In [ ]:

# --- Train Decision Tree ---
print("\nTraining Decision Tree...")

# We limit max_depth=3 so the tree is readable in the plot.
# If you want the full model for accuracy, remove max_depth (but it will be hard to plot).
dt_clf = DecisionTreeClassifier(max_depth=3, random_state=42, criterion='entropy')
dt_clf.fit(X_data, y_data)

# Evaluate Decision Tree
dt_acc = dt_clf.score(X_data, y_data)
print(f"Decision Tree Accuracy (max_depth=3): {dt_acc:.4f}")

# --- Visualize the Tree ---
class_names = ['Ordered', 'Disordered']

plt.figure(figsize=(20, 10))  # Large figure size for clarity
plot_tree(dt_clf,
          feature_names=AMINO_ACIDS,
          class_names=class_names,
          filled=True,          # Colors the nodes by class
          rounded=True,         # Rounds the corners of the nodes
          fontsize=10)

plt.title("Decision Tree for Protein Disorder Prediction (Top 3 Levels)")
plt.show()

Well this again only predicts ordered residues, we need to balance classes again

In [ ]:
# --- Train Decision Tree ---
print("\nTraining Decision Tree...")

# We limit max_depth=3 so the tree is readable in the plot.
# If you want the full model for accuracy, remove max_depth (but it will be hard to plot).
dt_clf = DecisionTreeClassifier(max_depth=3, random_state=42, class_weight='balanced', criterion='entropy')
dt_clf.fit(X_data, y_data)

# Evaluate Decision Tree
dt_acc = dt_clf.score(X_data, y_data)
print(f"Decision Tree Accuracy (max_depth=3): {dt_acc:.4f}")

# --- Visualize the Tree ---
class_names = ['Ordered', 'Disordered']

plt.figure(figsize=(20, 10))  # Large figure size for clarity
plot_tree(dt_clf,
          feature_names=AMINO_ACIDS,
          class_names=class_names,
          filled=True,          # Colors the nodes by class
          rounded=True,         # Rounds the corners of the nodes
          fontsize=10)

plt.title("Decision Tree for Protein Disorder Prediction (Top 3 Levels)")
plt.show()

### Step 8: Interpreting the Tree
If you look at the plot above, you can see exactly how the model makes decisions.
* **Root Node:** The top box is the most important feature (e.g., is the frequency of Isoleucine < 0.017?).
* **Branches:** The tree splits the data into smaller and smaller buckets. Left arrows are 'True', right arrows are 'False'
* **Leaves:** The final boxes tells us the prediction (Disordered or Ordered).
* **Gini:** A measure of "impurity". If Gini=0, that box contains only one class (perfect separation).

### Step 7: Neural Networks (Linear regression)

**The Architecture:**
* **Input Layer:** Our 20 amino acid frequencies.
* **Output Layer:** The final probability of being disordered.

Balancing the dataset is our responsibility now, we will also downsample each class, so it is faster to execute the code.

In [ ]:
def balance_classes(X, y, n_samples=None):
    """
    Balances the dataset by sampling from both classes.

    Arguments:
    X -- Feature matrix
    y -- Label vector
    n_samples -- (int) Number of samples to take from EACH class.
                 If None, defaults to the size of the minority class.

    Returns:
    X_bal, y_bal -- The balanced dataset
    """
    # Get indices for each class
    idx_0 = np.where(y == 0)[0]
    idx_1 = np.where(y == 1)[0]

    # Determine target sample size
    if n_samples is None:
        # Default: Downsample majority to match minority
        n_samples = min(len(idx_0), len(idx_1))

    # Safety check: Cannot sample more than what exists (unless we allow replacement)
    if n_samples > len(idx_0):
        print(f"Warning: Requested {n_samples} for Class 0 but only have {len(idx_0)}. Capping at {len(idx_0)}.")
        n_samples_0 = len(idx_0)
    else:
        n_samples_0 = n_samples

    if n_samples > len(idx_1):
        print(f"Warning: Requested {n_samples} for Class 1 but only have {len(idx_1)}. Capping at {len(idx_1)}.")
        n_samples_1 = len(idx_1)
    else:
        n_samples_1 = n_samples

    # Randomly Sample
    idx_0_selected = np.random.choice(idx_0, size=n_samples_0, replace=False)
    idx_1_selected = np.random.choice(idx_1, size=n_samples_1, replace=False)

    # Combine and Shuffle
    all_indices = np.concatenate([idx_0_selected, idx_1_selected])

    # Shuffle indices so classes are mixed (important for training!)
    np.random.shuffle(all_indices)

    return X[all_indices], y[all_indices]


X_bal, y_bal = balance_classes(X_data, y_data, n_samples=1000)
print(f"Auto-balanced shapes: {X_bal.shape}, {y_bal.shape}")

X_train, X_test, y_train, y_test = train_test_split(
    X_bal, y_bal, test_size=0.2, random_state=42, stratify=y_bal
)

In [ ]:
def sigmoid(z):
    """
    Compute the sigmoid of z
    Arguments:
    z -- A scalar or numpy array of any size.
    Return:
    s -- sigmoid(z)
    """
    return 1 / (1 + np.exp(-z))


def propagate(w, b, X, Y):
    m = X.shape[1]

    # Forward

    A = sigmoid(np.dot(w.T, X) + b)
    cost = (-1 / m) * np.sum(Y * np.log(A) + (1 - Y) * np.log(1 - A))

    # Backward

    dw = (1 / m) * np.dot(X, (A - Y).T)
    db = (1 / m) * np.sum(A - Y)

    cost = np.squeeze(np.array(cost))

    grads = {"dw": dw,
             "db": db}

    return grads, cost


def optimize(w, b, X, Y, num_iterations=100, learning_rate=0.009, print_cost=False):
    costs = []

    for i in range(num_iterations):

        grads, cost = propagate(w, b, X, Y)

        # Retrieve derivatives from grads
        dw = grads["dw"]
        db = grads["db"]

        # Update rule

        w = w - dw * learning_rate
        b = b - db * learning_rate

        # Record the costs
        if i % 100 == 0:
            costs.append(cost)

            # Print the cost every 100 training iterations
            if print_cost:
                print("Cost after iteration %i: %f" % (i, cost))

    params = {"w": w,
              "b": b}

    grads = {"dw": dw,
             "db": db}

    return params, grads, costs


def predict(w, b, X):
    w = w.reshape(X.shape[0], 1)
    A = sigmoid(np.dot(w.T, X) + b)
    Y_prediction = np.where(A > 0.5, 1, 0)
    return Y_prediction

In [ ]:
# --- Reshape Data for the Custom Network ---
# Our 'propagate' function expects X to be (features, m_samples)
# Scikit-learn provides X as (m_samples, features) -> We need to Transpose (.T)
train_set_x = X_train.T
test_set_x = X_test.T

# Our 'propagate' function expects Y to be (1, m_samples)
# Scikit-learn provides y as (m_samples,) -> We need to reshape
train_set_y = y_train.reshape(1, y_train.shape[0])
test_set_y = y_test.reshape(1, y_test.shape[0])

print(f"Original X shape: {X_train.shape}")
print(f"New X shape:      {train_set_x.shape}") # Should be (20, ~1.6 million)
print(f"New Y shape:      {train_set_y.shape}") # Should be (1, ~1.6 million)

In [ ]:
# --- Initialize Parameters ---
dim = train_set_x.shape[0] # This should be 20 (amino acids)
w = np.zeros((dim, 1))
b = 0.0

# --- Train the Model ---
print("\nTraining custom Neural Network...")
# Note: num_iterations is set low (500) for speed, increase to 2000+ for better results
# Learning rate might need tuning.
params, grads, costs = optimize(w, b, train_set_x, train_set_y,
                                num_iterations=1000,
                                learning_rate=5,
                                print_cost=True)

# Retrieve learned parameters
w_learned = params["w"]
b_learned = params["b"]

In [ ]:
# --- Prediction ---
print("\nPredictions:")

# Predict on Train set
Y_prediction_train = predict(w_learned, b_learned, train_set_x)
train_acc = 100 - np.mean(np.abs(Y_prediction_train - train_set_y)) * 100
print(f"Train accuracy: {train_acc:.2f}%")

# Predict on Test set
Y_prediction_test = predict(w_learned, b_learned, test_set_x)
test_acc = 100 - np.mean(np.abs(Y_prediction_test - test_set_y)) * 100
print(f"Test accuracy:  {test_acc:.2f}%")

# --- Visualize Cost Reduction ---
plt.figure(figsize=(6, 4))
plt.plot(costs)
plt.ylabel('Cost')
plt.xlabel('Iterations (per hundreds)')
plt.title("Learning Curve (Cost Reduction)")
plt.show()

### Step 7: Neural Networks (Deep Learning)
Finally, we will try a **Neural Network**. While Decision Trees are "rigid" (square cuts), Neural Networks can learn smooth, complex boundaries.

**The Architecture:**
* **Input Layer:** Our 20 amino acid frequencies.
* **Hidden Layer (`n_h`):** We are transforming the input into a new "representation."
    * `n_h = 2` means we compress the information into 2 abstract features.
    * Increasing this number allows the model to learn more complex patterns but increases the risk of memorizing the data (overfitting).
* **Output Layer:** The final probability of being disordered.

In [ ]:
def sigmoid(z):
    """
    Compute the sigmoid of z
    Arguments:
    z -- A scalar or numpy array of any size.
    Return:
    s -- sigmoid(z)
    """
    return 1 / (1 + np.exp(-z))


def layer_sizes(X, Y):
    """
    Arguments:
    X -- input dataset of shape (input size, number of examples)
    Y -- labels of shape (output size, number of examples)

    Returns:
    n_x -- the size of the input layer
    n_h -- the size of the hidden layer
    n_y -- the size of the output layer
    """

    n_x = X.shape[0]
    n_h = 4
    n_y = Y.shape[0]

    return n_x, n_h, n_y


def initialize_parameters(n_x, n_h, n_y):
    """
    Argument:
    n_x -- size of the input layer
    n_h -- size of the hidden layer
    n_y -- size of the output layer

    Returns:
    params -- python dictionary containing your parameters:
                    W1 -- weight matrix of shape (n_h, n_x)
                    b1 -- bias vector of shape (n_h, 1)
                    W2 -- weight matrix of shape (n_y, n_h)
                    b2 -- bias vector of shape (n_y, 1)
    """

    W1 = np.random.randn(n_h, n_x) * 0.01
    b1 = np.zeros((n_h, 1))
    W2 = np.random.randn(n_y, n_h) * 0.01
    b2 = np.zeros((n_y, 1))

    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2}

    return parameters


def forward_propagation(X, parameters):
    """
    Argument:
    X -- input data of size (n_x, m)
    parameters -- python dictionary containing your parameters (output of initialization function)

    Returns:
    A2 -- The sigmoid output of the second activation
    cache -- a dictionary containing "Z1", "A1", "Z2" and "A2"
    """
    # Retrieve each parameter from the dictionary "parameters"

    W1 = parameters['W1']
    b1 = parameters['b1']
    W2 = parameters['W2']
    b2 = parameters['b2']

    # Implement Forward Propagation to calculate A2 (probabilities)

    Z1 = np.dot(W1, X) + b1

    A1 = np.tanh(Z1)
    Z2 = np.dot(W2, A1) + b2
    A2 = sigmoid(Z2)

    assert (A2.shape == (1, X.shape[1]))

    cache = {"Z1": Z1,
             "A1": A1,
             "Z2": Z2,
             "A2": A2}

    return A2, cache


def compute_cost(A2, Y):
    """
    Computes the cross-entropy cost given in equation (13)

    Arguments:
    A2 -- The sigmoid output of the second activation, of shape (1, number of examples)
    Y -- "true" labels vector of shape (1, number of examples)

    Returns:
    cost -- cross-entropy cost given equation (13)

    """

    # Compute the cross-entropy cost

    logprobs = Y * np.log(A2) + (1 - Y) * np.log(1 - A2)
    cost = -np.sum(logprobs) / Y.shape[1]

    cost = float(np.squeeze(cost))  # makes sure cost is the dimension we expect.
    # E.g., turns [[17]] into 17

    return cost


def backward_propagation(parameters, cache, X, Y):
    """
    Implement the backward propagation using the instructions above.

    Arguments:
    parameters -- python dictionary containing our parameters
    cache -- a dictionary containing "Z1", "A1", "Z2" and "A2".
    X -- input data of shape (2, number of examples)
    Y -- "true" labels vector of shape (1, number of examples)

    Returns:
    grads -- python dictionary containing your gradients with respect to different parameters
    """
    m = X.shape[1]

    # First, retrieve W1 and W2 from the dictionary "parameters".

    W1 = parameters["W1"]
    W2 = parameters["W2"]

    # Retrieve also A1 and A2 from dictionary "cache".

    A1 = cache['A1']
    A2 = cache['A2']

    # Backward propagation: calculate dW1, db1, dW2, db2.

    dZ2 = A2 - Y
    dW2 = (1 / m) * np.dot(dZ2, A1.T)
    db2 = (1 / m) * np.sum(dZ2, axis=1, keepdims=True)
    dZ1 = np.dot(W2.T, dZ2) * (1 - np.power(A1, 2))
    dW1 = (1 / m) * np.dot(dZ1, X.T)
    db1 = (1 / m) * np.sum(dZ1, axis=1, keepdims=True)

    grads = {"dW1": dW1,
             "db1": db1,
             "dW2": dW2,
             "db2": db2}

    return grads


def update_parameters(parameters, grads, learning_rate=1.2):
    """
    Updates parameters using the gradient descent update rule given above

    Arguments:
    parameters -- python dictionary containing your parameters
    grads -- python dictionary containing your gradients

    Returns:
    parameters -- python dictionary containing your updated parameters
    """
    # Retrieve a copy of each parameter from the dictionary "parameters". Use copy.deepcopy(...) for W1 and W2

    W1 = copy.deepcopy(parameters['W1'])
    b1 = parameters['b1']
    W2 = copy.deepcopy(parameters['W2'])
    b2 = parameters['b2']

    # Retrieve each gradient from the dictionary "grads"

    dW1 = grads['dW1']
    db1 = grads['db1']
    dW2 = grads['dW2']
    db2 = grads['db2']

    # Update rule for each parameter

    W1 = W1 - dW1 * learning_rate
    b1 = b1 - db1 * learning_rate
    W2 = W2 - dW2 * learning_rate
    b2 = b2 - db2 * learning_rate

    parameters = {"W1": W1,
                  "b1": b1,
                  "W2": W2,
                  "b2": b2}

    return parameters


def nn_model(X, Y, n_h, num_iterations=10000, print_cost=False):
    """
    Arguments:
    X -- dataset of shape (2, number of examples)
    Y -- labels of shape (1, number of examples)
    n_h -- size of the hidden layer
    num_iterations -- Number of iterations in gradient descent loop
    print_cost -- if True, print the cost every 1000 iterations

    Returns:
    parameters -- parameters learnt by the model. They can then be used to predict.
    """

    np.random.seed(3)
    n_x = layer_sizes(X, Y)[0]
    n_y = layer_sizes(X, Y)[2]

    # Initialize parameters

    parameters = initialize_parameters(n_x, n_h, n_y)

    # Loop (gradient descent)

    for i in range(0, num_iterations):

        # Forward propagation. Inputs: "X, parameters". Outputs: "A2, cache".
        A2, cache = forward_propagation(X, parameters)

        # Cost function. Inputs: "A2, Y". Outputs: "cost".
        cost = compute_cost(A2, Y)

        # Backpropagation. Inputs: "parameters, cache, X, Y". Outputs: "grads".
        grads = backward_propagation(parameters, cache, X, Y)

        # Gradient descent parameter update. Inputs: "parameters, grads". Outputs: "parameters".
        parameters = update_parameters(parameters, grads)

        # Print the cost every 1000 iterations
        if print_cost and i % 1000 == 0:
            print("Cost after iteration %i: %f" % (i, cost))

    return parameters


def nn_model_with_tracking(X_train, Y_train, X_test, Y_test, n_h, num_iterations=10000, print_cost=False):
    """
    Arguments:
    X_train -- training data
    Y_train -- training labels
    X_test -- test data (for validation tracking)
    Y_test -- test labels
    n_h -- size of the hidden layer
    num_iterations -- Number of iterations in gradient descent loop
    print_cost -- if True, print the cost every 1000 iterations

    Returns:
    parameters -- parameters learnt by the model
    train_costs -- list of training costs
    test_costs -- list of test costs
    """

    np.random.seed(3)
    n_x = layer_sizes(X_train, Y_train)[0]
    n_y = layer_sizes(X_train, Y_train)[2]

    # Initialize parameters
    parameters = initialize_parameters(n_x, n_h, n_y)

    # Lists to keep track of cost
    train_costs = []
    test_costs = []

    # Loop (gradient descent)
    for i in range(0, num_iterations):

        # 1. Forward propagation (Train)
        A2, cache = forward_propagation(X_train, parameters)

        # 2. Cost function (Train)
        cost = compute_cost(A2, Y_train)

        # 3. Backpropagation
        grads = backward_propagation(parameters, cache, X_train, Y_train)

        # 4. Update parameters
        parameters = update_parameters(parameters, grads)

        # --- TRACKING ---
        # Record every 100 iterations to save time/memory
        if i % 100 == 0:
            train_costs.append(cost)

            # Calculate Test Cost (Forward pass only, no backprop needed)
            A2_test, _ = forward_propagation(X_test, parameters)
            cost_test = compute_cost(A2_test, Y_test)
            test_costs.append(cost_test)

            # Print every 1000 iterations
            if print_cost and i % 1000 == 0:
                print(f"Iteration {i}: Train Cost = {cost:.6f} | Test Cost = {cost_test:.6f}")

    return parameters, train_costs, test_costs


def predict(parameters, X):
    """
    Using the learned parameters, predicts a class for each example in X

    Arguments:
    parameters -- python dictionary containing your parameters
    X -- input data of size (n_x, m)

    Returns
    predictions -- vector of predictions of our model (red: 0 / blue: 1)
    """

    # Computes probabilities using forward propagation, and classifies to 0/1 using 0.5 as the threshold.

    A2, cache = forward_propagation(X, parameters)
    predictions = np.where(A2 > 0.5, 1, 0)

    return predictions


train_set_x = X_train.T
train_set_y = y_train.reshape(1, y_train.shape[0]) # Reshape to (1, m)

test_set_x = X_test.T
test_set_y = y_test.reshape(1, y_test.shape[0])

print(f"Network input shape: {train_set_x.shape}") # Should be (20, N)
print(f"Network label shape: {train_set_y.shape}") # Should be (1, N)

In [ ]:
# --- Train the Model with Tracking ---
print("\nTraining 2-Layer Network with Train/Test Tracking...")

# Ensure inputs are the correct shape (n_features, m_samples)
# If you haven't transposed them yet, uncomment these lines:
# train_set_x = X_train.T
# train_set_y = y_train.reshape(1, -1)
# test_set_x = X_test.T
# test_set_y = y_test.reshape(1, -1)

parameters, train_costs, test_costs = nn_model_with_tracking(
    train_set_x, train_set_y,
    test_set_x, test_set_y,
    n_h=2,
    num_iterations=50000,
    print_cost=True
)

# --- Evaluate ---
print("\nEvaluating...")
predictions_test = predict(parameters, test_set_x)
accuracy = float((np.dot(test_set_y, predictions_test.T) + np.dot(1 - test_set_y, 1 - predictions_test.T)) / float(test_set_y.size) * 100)
print(f'Final Test Accuracy: {accuracy:.2f} %')

# --- Plotting the Loss Curves ---
plt.figure(figsize=(10, 6))
plt.plot(train_costs, label='Train Cost', color='blue')
plt.plot(test_costs, label='Test Cost', color='orange', linestyle='--')

plt.ylabel('Cost (Cross-Entropy)')
plt.xlabel('Iterations (per hundreds)')
plt.title("Learning Curve: Train vs Test Loss")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

### Learning Curves (Are we learning?)
Training a Neural Network is an iterative process. We make a prediction, measure the error (**Cost**), and slightly adjust the weights to reduce that error. We do this thousands of times.

**How to read the Plot:**
* **Blue Line (Train Cost):** Should always go down. This means the model is learning the training data.
* **Orange Line (Test Cost):** This is the crucial one!
    * **Good:** It goes down and stays low.
    * **Overfitting:** It goes down for a while, then starts moving **UP**. This means the model is memorizing the training examples but getting worse at generalizing to new proteins.

### Conclusion
We moved from simple linear models to complex Neural Networks.

**Takeaways:**
1.  **Data Matters:** The imbalance (mostly Ordered residues) made accuracy a misleading metric.
2.  **Features Matter:** Using a "Sliding Window" allowed us to capture the neighborhood context of each amino acid.
3.  **Complexity Trade-off:** More complex models (like Neural Nets) aren't always better if we don't have enough data; they might just overfit.